# Multiclass Classification with pytorch
* Glass_Identification/Glass.csv


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir("D:\Machine Learning\Datasets\Cases\Glass_Identification")
import torch 
import torch.nn as nn
from torchsummary import summary
#  sklearn imports 
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, r2_score

In [8]:
glass = pd.read_csv('Glass.csv')
dum_gls = pd.get_dummies(glass)
X = dum_gls.iloc[:,:-6]
y = dum_gls.iloc[:,-6:]

In [9]:
scaler = MinMaxScaler()
X_train, X_test, y_train, y_test = train_test_split(X,y.values, random_state =25, test_size = 0.3, stratify=glass['Type'])
X_scl_trn = scaler.fit_transform(X_train)
X_scl_tst = scaler.transform(X_test)
X_scl_trn.shape

(149, 9)

In [10]:
X_torch = torch.from_numpy(X_scl_trn)
y_torch = torch.from_numpy(y_train)
print(X_torch.size())
print(y_torch.size())

torch.Size([149, 9])
torch.Size([149, 6])


In [14]:
torch.manual_seed(25)
model = nn.Sequential(nn.Linear(in_features=X_scl_trn.shape[1], out_features=18),
                    
                    nn.ReLU()
                    ,nn.Linear(in_features=18, out_features=6)
                    )
summary(model, (1,X_scl_trn.shape[1]))                    

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                [-1, 1, 18]             180
              ReLU-2                [-1, 1, 18]               0
            Linear-3                 [-1, 1, 6]             114
Total params: 294
Trainable params: 294
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.05)
optimizer

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.05
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

#### Generating Predictions with default weights

In [18]:
y_pred = model(X_torch.float()) # Weighted Sums aka lin_outputs not actual probas
y_pred[:3]

tensor([[ 0.1255,  0.3045, -0.0804,  0.0280,  0.1467, -0.3462],
        [ 0.1773,  0.2718, -0.0696, -0.0141,  0.1395, -0.3197],
        [ 0.1468,  0.2995, -0.0838,  0.0472,  0.1369, -0.3467]],
       grad_fn=<SliceBackward0>)

In [17]:
print(y_torch.shape)

torch.Size([149, 6])


In [20]:
loss = criterion(y_pred, y_torch.float())
loss

tensor(1.7186, grad_fn=<DivBackward1>)

In [21]:
for epoch in np.arange(0,1000):
    # Forward Pass:  Compute predicted y by passing x to the model
    y_pred_prob = model(X_torch.float()) # Inferencing of X
    # Compute and print loss
    loss = criterion(y_pred_prob, y_torch.float())
    if epoch % 100 == 0:
        print('Epoch', epoch+1, 'Loss:', loss.item())
    # Zero Gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad() # all gradients are equaled to 0
    #perform a backward pass (backpropogation)
    loss.backward()
    #
    optimizer.step()


Epoch 1 Loss: 1.7186344861984253
Epoch 101 Loss: 1.46933913230896
Epoch 201 Loss: 1.3758621215820312
Epoch 301 Loss: 1.2651134729385376
Epoch 401 Loss: 1.1757515668869019
Epoch 501 Loss: 1.1176071166992188
Epoch 601 Loss: 1.075873851776123
Epoch 701 Loss: 1.0433498620986938
Epoch 801 Loss: 1.0167316198349
Epoch 901 Loss: 0.9942197799682617


In [24]:
X_torch_test = torch.from_numpy(X_scl_tst)
type(X_torch_test)

torch.Tensor

In [26]:
lin_output = model(X_torch_test.float()) # Equivalent predict_proba
type(lin_output)


torch.Tensor

In [30]:
y_wt_sum = model(X_torch_test.float())
softmax = nn.Softmax(dim=1)
pred_proba = softmax(y_wt_sum)
pred_proba

tensor([[5.8695e-01, 2.9242e-01, 7.6382e-03, 1.4663e-03, 1.1330e-02, 1.0019e-01],
        [4.1819e-01, 4.1192e-01, 2.3809e-02, 1.0379e-02, 3.0064e-02, 1.0564e-01],
        [4.4359e-01, 4.0753e-01, 1.8132e-02, 6.7682e-03, 2.3336e-02, 1.0064e-01],
        [4.3652e-01, 4.1361e-01, 1.8634e-02, 6.8582e-03, 2.3824e-02, 1.0055e-01],
        [4.2667e-01, 4.0766e-01, 2.2095e-02, 9.9252e-03, 2.8931e-02, 1.0472e-01],
        [6.0046e-01, 2.8183e-01, 7.6809e-03, 2.1972e-03, 1.2444e-02, 9.5389e-02],
        [5.7995e-01, 2.9570e-01, 8.8843e-03, 2.1145e-03, 1.3245e-02, 1.0011e-01],
        [4.9613e-01, 3.5449e-01, 1.5490e-02, 6.2269e-03, 2.2830e-02, 1.0483e-01],
        [4.5836e-01, 3.9298e-01, 1.7192e-02, 6.7154e-03, 2.3222e-02, 1.0153e-01],
        [3.6741e-01, 4.3580e-01, 3.2678e-02, 1.6015e-02, 3.9746e-02, 1.0835e-01],
        [1.2470e-01, 3.4742e-01, 1.0861e-01, 2.3353e-01, 1.1198e-01, 7.3766e-02],
        [4.9983e-01, 3.6921e-01, 1.2577e-02, 3.7587e-03, 1.7100e-02, 9.7525e-02],
        [3.7520e

In [32]:
y_pred_scl = pred_proba.detach().numpy()
y_pred_scl[:3]

array([[0.58695316, 0.29242018, 0.0076382 , 0.00146627, 0.01132956,
        0.10019254],
       [0.41818526, 0.4119216 , 0.02380899, 0.01037906, 0.03006386,
        0.10564117],
       [0.44359264, 0.40753236, 0.0181323 , 0.00676815, 0.02333637,
        0.10063822]], dtype=float32)

In [33]:
y_pred = np.argmax(pred_proba.detach().numpy(), axis = 1)
y_pred

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 3, 3, 1, 3, 1, 0, 1, 1, 0,
       0, 0, 0, 0, 0, 0, 3, 0, 1, 1, 3, 1, 1, 0, 0, 3, 3, 1, 0, 0, 0, 0,
       3, 1, 1, 1, 0, 3, 1, 1, 0, 0, 1, 1, 3, 3, 0, 0, 3, 1, 1, 0, 1])

In [34]:
y_test_org = np.argmax(y_test, axis = 1)
print(accuracy_score(y_test_org,y_pred))

0.49230769230769234


In [36]:
log_loss(y_test_org, y_pred_prob)

0.22140717932941575